# Market Returns Around VIX Peaks (2016–2025)
## Replication & Extension of an Event-Study Framework

**Course:** FIN 559 | **Group 4**

---

### Research Question
How do U.S. equity markets behave in the 10 trading days before and after spikes in
the CBOE Volatility Index (VIX)? Does the pattern differ across **market segments**
(proxied by realized volatility) and **sectors**?

### Methodology Overview
1. Identify *local maxima* in VIX that exceed a minimum threshold (standard event-study design).
2. Align S&P 500 returns into a `[-10, +10]` trading-day event window around each peak.
3. Compute Average Abnormal Returns (AAR) and Cumulative AARs (CAR).
4. Extend cross-sectionally using **volatility-sorted portfolios** (as a size proxy)
   and GICS **sector portfolios** from current S&P 500 constituents.

> **Data note:** All price data is downloaded live from Yahoo Finance via `yfinance`.  
> Results may vary slightly with each run because `END_DATE` defaults to today.  
> To pin results, set `END_DATE` to a fixed string in the *Configuration* cell below.

---
## 1. Imports & Configuration

In [ ]:
import os
import json as _json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # headless backend — safe for all environments
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive')

In [ ]:
# ── Study parameters ─────────────────────────────────────────────────────
START_DATE = '2016-01-01'
END_DATE   = None   # Set to e.g. '2025-12-31' to pin results; None = today

# VIX peak detection
PEAK_WINDOW   = 5    # local-max half-window (scans 2*PEAK_WINDOW+1 days)
VIX_THRESHOLD = 25   # minimum VIX level to qualify as a peak

# Event study window
PRE_DAYS  = 10
POST_DAYS = 10

# Output directory for saved figures
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Resolve end date
if END_DATE is None:
    END_DATE = pd.Timestamp.today().strftime('%Y-%m-%d')

print(f'Study period : {START_DATE}  →  {END_DATE}')
print(f'VIX threshold: {VIX_THRESHOLD} | event window: [{-PRE_DAYS}, +{POST_DAYS}] trading days')

---
## 2. Data Collection

We download daily closing prices for the VIX index and the S&P 500 (^GSPC) from Yahoo Finance.

In [ ]:
vix_df = yf.download('^VIX',  start=START_DATE, end=END_DATE,
                     auto_adjust=True, progress=False)
spx_df = yf.download('^GSPC', start=START_DATE, end=END_DATE,
                     auto_adjust=True, progress=False)

print(f'VIX rows : {len(vix_df):,}')
print(f'S&P rows : {len(spx_df):,}')

In [ ]:
# Extract closing prices and compute S&P 500 daily returns
vix = vix_df['Close'].squeeze()
vix.name = 'VIX'

spx_close = spx_df['Close'].squeeze()
spx_close.name = 'SPX'

spx_ret = spx_close.pct_change().rename('SPX_ret')

data = pd.concat([vix, spx_ret], axis=1).dropna()
print(f'Combined dataset: {len(data):,} trading days')
data.head(3)

---
## 3. VIX Peak Detection

A trading day is classified as a **VIX peak** if:
1. Its VIX reading equals the rolling maximum over a `±PEAK_WINDOW`-day window (i.e., it is a local maximum), **and**
2. The VIX reading is ≥ `VIX_THRESHOLD` (25 by default).

This avoids trivially small volatility blips and captures meaningful stress episodes.

In [ ]:
data['rolling_max'] = data['VIX'].rolling(
    window=2 * PEAK_WINDOW + 1, center=True
).max()

data['is_peak'] = (
    (data['VIX'] == data['rolling_max']) &
    (data['VIX'] >= VIX_THRESHOLD)
)
vix_peaks = data.index[data['is_peak']]

print(f'Detected {len(vix_peaks)} VIX peaks ≥ {VIX_THRESHOLD} '
      f'from {START_DATE} to {END_DATE}:\n')

peaks_table = data.loc[vix_peaks, ['VIX']].sort_index(ascending=False)
print(peaks_table.head(15).to_string())

---
## 4. Event Study: S&P 500 Index Returns

For each VIX peak we extract a `[–PRE_DAYS, +POST_DAYS]` window of S&P 500 daily returns
(using *trading* days, not calendar days). We then average across all peak events to produce:

- **AAR** — Average Abnormal Return on each event-day
- **CAR** — Cumulative AAR from the start of the window

*Note: because all peaks are treated symmetrically and no market-model benchmark is subtracted,
'abnormal' here simply means the raw return averaged across events.*

In [ ]:
def event_window(series, peaks, pre=PRE_DAYS, post=POST_DAYS):
    """
    Build an event matrix and return AAR, CAR, and the day-offset array.

    Parameters
    ----------
    series : pd.Series  Daily return series (date-indexed).
    peaks  : DatetimeIndex  Event dates.
    pre    : int  Trading days before each peak.
    post   : int  Trading days after each peak.

    Returns
    -------
    AAR  : np.ndarray | None
    CAR  : np.ndarray | None
    days : np.ndarray  Integer offsets [-pre, ..., +post].
    """
    mats = []
    for peak in peaks:
        window = series.loc[
            peak - pd.Timedelta(days=pre * 2):
            peak + pd.Timedelta(days=post * 2)
        ]
        if len(window) >= pre + post + 1:
            mats.append(window.iloc[-(pre + post + 1):].values)
    if not mats:
        return None, None, np.arange(-pre, post + 1)
    mats = np.vstack(mats)
    AAR = mats.mean(axis=0)
    CAR = AAR.cumsum()
    return AAR, CAR, np.arange(-pre, post + 1)

In [ ]:
# Build event matrix for the S&P 500 index
event_rows = []
for peak in vix_peaks:
    window_data = data.loc[
        peak - pd.Timedelta(days=PRE_DAYS * 2):
        peak + pd.Timedelta(days=POST_DAYS * 2)
    ]
    if len(window_data) < PRE_DAYS + POST_DAYS + 1:
        continue
    event_rows.append(window_data['SPX_ret'].iloc[-(PRE_DAYS + POST_DAYS + 1):].values)

print(f'Events with complete windows: {len(event_rows)} / {len(vix_peaks)}')

In [ ]:
event_matrix = np.vstack(event_rows)
AAR  = event_matrix.mean(axis=0)
CAR  = AAR.cumsum()
days = np.arange(-PRE_DAYS, POST_DAYS + 1)

print(f'Event matrix shape: {event_matrix.shape}  (events × trading days)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(days, AAR * 100, label='Average Daily Return (%)')
ax.axvline(0, color='red', linestyle='--', label='VIX Peak (Day 0)')
ax.set_title(
    f'Average S&P 500 Return Around VIX Peaks ({START_DATE[:4]}–{END_DATE[:4]})',
    fontsize=13
)
ax.set_xlabel('Trading Days Relative to VIX Peak')
ax.set_ylabel('Return (%)')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_aar_spx.png'), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(days, CAR * 100, color='steelblue', lw=2, label='Cumulative Return (%)')
ax.axvline(0, color='red', linestyle='--', label='VIX Peak (Day 0)')
ax.set_title(
    f'Cumulative S&P 500 Return Around VIX Peaks ({START_DATE[:4]}–{END_DATE[:4]})',
    fontsize=13
)
ax.set_xlabel('Trading Days Relative to VIX Peak')
ax.set_ylabel('Cumulative Return (%)')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig2_car_spx.png'), dpi=150)
plt.show()

In [ ]:
pre_mean  = AAR[days < 0].mean()
post_mean = AAR[days > 0].mean()

print('S&P 500 Index — Average Daily Return')
print(f'  Before VIX peaks : {pre_mean  * 100:.3f}%')
print(f'  After  VIX peaks : {post_mean * 100:.3f}%')

---
## 5. Cross-Sectional Extensions

We extend the analysis to individual S&P 500 constituents to test whether the pre/post-peak
return asymmetry varies by firm size or sector.

### 5.1 S&P 500 Constituents — Data Collection

In [ ]:
# Download current S&P 500 constituents list from a public GitHub dataset
sp500 = pd.read_csv(
    'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv'
)
print(f'Constituent tickers loaded: {len(sp500)}')
sp500.head(3)

In [ ]:
# Yahoo Finance uses '-' instead of '.' in ticker symbols (e.g. BRK.B -> BRK-B)
sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-', regex=False)

In [ ]:
# WBA was delisted mid-study; exclude to avoid spurious download errors
BAD_TICKERS = ['WBA']
tickers = [t for t in sp500['Symbol'].unique().tolist() if t not in BAD_TICKERS]
print(f'Downloading price data for {len(tickers)} tickers — this may take ~1–2 minutes …')

raw = yf.download(tickers, start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=False)

if raw.empty:
    raise RuntimeError('yf.download returned no data for S&P 500 tickers.')

# Extract closing prices robustly (MultiIndex layout depends on yfinance version)
if isinstance(raw.columns, pd.MultiIndex):
    lvl0 = raw.columns.get_level_values(0)
    price_col = 'Adj Close' if 'Adj Close' in lvl0 else 'Close'
    prices = raw.xs(price_col, axis=1, level=0)
else:
    price_col = 'Adj Close' if 'Adj Close' in raw.columns else 'Close'
    prices = raw[[price_col]]

print(f'Price data shape: {prices.shape}  (trading days × tickers)')

In [ ]:
# Drop tickers with no data, then compute daily returns
prices  = prices.dropna(axis=1, how='all')
returns = prices.pct_change().dropna(how='all')

print(f'Returns shape: {returns.shape}  (trading days × usable tickers)')
print(f'First 5 columns: {list(returns.columns[:5])}')

### 5.2 Volatility-Based Size Proxy

We do **not** have direct market-capitalisation data here. Instead, we use each stock's
**realized daily return volatility** as a proxy:

> Low volatility ≈ large, well-established companies ('large-cap proxy')  
> High volatility ≈ smaller, more volatile companies ('small-cap proxy')

Stocks are split at the cross-sectional median volatility into two equal-sized portfolios.

**Caveat:** Volatility and market capitalisation are correlated but not identical.
This classification is a common empirical shortcut when capitalisation data is unavailable,
but results should not be interpreted as a formal size-sorted portfolio study.

In [ ]:
# Classify stocks into two groups by realized return volatility
volatility  = returns.std().sort_values()
median_vol  = volatility.median()

low_vol_tickers  = volatility[volatility <= median_vol].index   # large-cap proxy
high_vol_tickers = volatility[volatility >  median_vol].index   # small-cap proxy

print(f'Low-volatility  (large-cap proxy): {len(low_vol_tickers)} stocks')
print(f'High-volatility (small-cap proxy): {len(high_vol_tickers)} stocks')
print(f'Median daily vol: {median_vol:.4f}  ({median_vol*100:.2f}%)')

### 5.3 Sector Information

Sector labels are fetched from Yahoo Finance via individual `yf.Ticker` calls.
This step makes ~500 HTTP requests and may take **5–15 minutes** depending on network speed.
If a ticker's sector cannot be retrieved it is labelled `'Unknown'`.

In [ ]:
SECTOR_CACHE = os.path.join(os.path.dirname(os.path.abspath('.')),
                            'sector_cache.json')
SECTOR_CACHE = 'sector_cache.json'   # saved alongside the notebook

if os.path.exists(SECTOR_CACHE):
    with open(SECTOR_CACHE) as _f:
        sector_info = _json.load(_f)
    print(f'Loaded sector info from cache ({len(sector_info)} tickers)')
else:
    print(f'Fetching sector info for {len(returns.columns)} tickers '
          f'via yfinance — this may take 5-15 min on first run ...')
    sector_info = {}
    for t in returns.columns:
        try:
            tkr = yf.Ticker(t)
            sector_info[t] = tkr.info.get('sector', 'Unknown')
        except Exception:
            sector_info[t] = 'Unknown'
    with open(SECTOR_CACHE, 'w') as _f:
        _json.dump(sector_info, _f)
    print(f'Sector info cached to {SECTOR_CACHE}')

meta = pd.DataFrame({'sector': sector_info})
print('Sector distribution:')
print(meta['sector'].value_counts().to_string())

In [ ]:
# Build equal-weighted portfolios
port_low_vol  = returns[low_vol_tickers].mean(axis=1)   # large-cap proxy
port_high_vol = returns[high_vol_tickers].mean(axis=1)  # small-cap proxy

# Build sector portfolios (require >= 5 stocks to reduce noise)
sector_ports = {}
for sector in meta['sector'].dropna().unique():
    sector_tickers = [t for t in meta[meta['sector'] == sector].index
                      if t in returns.columns]
    if len(sector_tickers) > 5:
        sector_ports[sector] = returns[sector_tickers].mean(axis=1)

print(f'Sector portfolios built: {list(sector_ports.keys())}')

---
## 6. Event Study: Volatility-Sorted Portfolios (Size Proxy)

We apply the same event-study framework to the two volatility-sorted portfolios
to test whether high-volatility (small-cap proxy) stocks exhibit a stronger mean-reversion
response after VIX peaks — consistent with the *overreaction hypothesis*.

In [ ]:
AAR_low,  CAR_low,  days = event_window(port_low_vol,  vix_peaks)
AAR_high, CAR_high, _    = event_window(port_high_vol, vix_peaks)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(days, CAR_low  * 100, lw=2, label='Low-Vol  (large-cap proxy)')
ax.plot(days, CAR_high * 100, lw=2, label='High-Vol (small-cap proxy)')
ax.axvline(0, color='red', linestyle='--', label='VIX Peak (Day 0)')
ax.set_title(
    f'Cumulative Returns Around VIX Peaks\n'
    f'by Volatility Group / Size Proxy ({START_DATE[:4]}–{END_DATE[:4]})',
    fontsize=12
)
ax.set_xlabel('Trading Days Relative to VIX Peak')
ax.set_ylabel('Cumulative Return (%)')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_car_size_proxy.png'), dpi=150)
plt.show()

In [ ]:
pre_low   = AAR_low [days < 0].mean()
post_low  = AAR_low [days > 0].mean()
pre_high  = AAR_high[days < 0].mean()
post_high = AAR_high[days > 0].mean()

print('Average daily returns by volatility group')
print(f'  Low-Vol  (large-cap proxy) : pre {pre_low  * 100:.3f}%  →  post {post_low  * 100:.3f}%')
print(f'  High-Vol (small-cap proxy) : pre {pre_high * 100:.3f}%  →  post {post_high * 100:.3f}%')

if post_high > post_low:
    print('\n→ High-volatility stocks rebound more strongly after VIX peaks,'
          ' consistent with the overreaction hypothesis.')
else:
    print('\n→ No clear volatility-group asymmetry in the post-peak rebound.')

---
## 7. Event Study: Sector Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for sector, port in sector_ports.items():
    _, CAR_sector, _ = event_window(port, vix_peaks)
    if CAR_sector is not None:
        ax.plot(days, CAR_sector * 100, lw=1.5, label=sector)
ax.axvline(0, color='red', linestyle='--', label='VIX Peak (Day 0)')
ax.set_title(
    f'Cumulative Returns Around VIX Peaks by Sector ({START_DATE[:4]}–{END_DATE[:4]})',
    fontsize=12
)
ax.set_xlabel('Trading Days Relative to VIX Peak')
ax.set_ylabel('Cumulative Return (%)')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_car_sectors.png'), dpi=150)
plt.show()

In [ ]:
sector_rows = []
for sector, port in sector_ports.items():
    AAR_sector, _, _ = event_window(port, vix_peaks)
    if AAR_sector is not None:
        pre  = AAR_sector[days < 0].mean()
        post = AAR_sector[days > 0].mean()
        sector_rows.append((sector, pre * 100, post * 100, (post - pre) * 100))

sector_summary = pd.DataFrame(
    sector_rows, columns=['Sector', 'Pre (%)', 'Post (%)', 'Delta Post-Pre (%)']
).sort_values('Delta Post-Pre (%)', ascending=False)

print('Sector reactions to VIX peaks (average daily returns, %):\n')
print(sector_summary.to_string(index=False, float_format='{:.4f}'.format))

---
## 8. Key Findings & Limitations

### Key Findings

1. **Pre/post reversal in the S&P 500 index:**  
   S&P 500 returns are negative on average in the 10 days *before* a VIX peak and positive
   in the 10 days *after*, suggesting the market tends to recover once fear reaches a local
   maximum — consistent with mean-reversion dynamics documented in the literature.

2. **Volatility-group asymmetry (size proxy):**  
   High-volatility stocks (small-cap proxy) experience larger drawdowns before peaks and
   stronger rebounds afterwards, broadly consistent with the overreaction hypothesis.
   Results should be interpreted cautiously because the grouping is volatility-based,
   not based on actual market-capitalisation data.

3. **Sector heterogeneity:**  
   Energy and Technology tend to show the largest post-peak reversals; defensive sectors
   (Utilities, Consumer Staples) show smaller swings in both directions.

---

### Limitations

| # | Limitation |
|---|------------|
| 1 | **No benchmark subtraction** — returns are raw, not market-model residuals. |
| 2 | **Survivorship bias** — the constituent list reflects the *current* S&P 500; firms delisted between 2016–2025 are excluded. |
| 3 | **Volatility ≠ market cap** — the size proxy is an approximation; formal size-sorted portfolios require capitalisation data. |
| 4 | **Overlapping event windows** — some peaks are close together, causing window overlap and potentially inflated significance. |
| 5 | **Live data dependency** — results depend on `yfinance` availability and may change as data is revised or the constituent list updates. |
| 6 | **Sector labels from yfinance** — labels reflect current sector assignments, not historical classifications. |

---

### Potential Extensions
- Replace the volatility proxy with actual market-cap data (e.g. from CRSP or Compustat).
- Apply a Fama-French three-factor or CAPM benchmark to compute genuine abnormal returns.
- Test statistical significance of the CAR using a standard event-study t-test.
- Explore sub-periods (pre/post COVID, rate-hike cycles) to assess structural breaks.
- Investigate VIX *troughs* to test asymmetry in investor response.